In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

c:\Users\gagan\Desktop\Mtech AI ML\Sem 2\Natural Language Processing (S1-26_AIMLZG530)\Assignment\venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


PyTorch version: 2.14.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [4]:
%pip install nltk numpy pandas matplotlib

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 5.9 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 5.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.6 MB 7.1 MB/s eta 0:00:02
   ---------- ----------------------------- 3.4/12.6 MB 8.0 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.6 MB 8.8 MB/s eta 0:00:01
   ------------------------- -------------- 8.1/12.6 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.6 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 11.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   -------------------- ------------------- 5.0/9.6 MB 23.9 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 26.5 MB/s eta 0:00:00
   -----------------


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import nltk
from nltk.corpus import treebank

In [6]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed set to:", SEED)

Random seed set to: 42


In [7]:
nltk.download("treebank")

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\gagan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\treebank.zip.


True

In [8]:
print("Number of sentences:", len(treebank.sents()))
print("Number of words:", len(treebank.words()))
print("Number of tagged words:", len(treebank.tagged_words()))
print("Number of tagged sentences:", len(treebank.tagged_sents()))

Number of sentences: 3914
Number of words: 100676
Number of tagged words: 100676
Number of tagged sentences: 3914


In [9]:
sentences = treebank.sents()

print("Number of sentences:", len(sentences))
print("\nFirst sentence:")
print(sentences[0])

Number of sentences: 3914

First sentence:
['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'Nov.', '29', '.']


In [11]:
num_sentences = len(sentences)
num_tokens = sum(len(sentence) for sentence in sentences)

print("Number of sentences:", num_sentences)
print("Total tokens:", num_tokens)
print("Average tokens per sentence:", num_tokens / num_sentences)

Number of sentences: 3914
Total tokens: 100676
Average tokens per sentence: 25.722023505365357


In [12]:
sentence_lengths = [len(sentence) for sentence in sentences]

print("Minimum sentence length:", min(sentence_lengths))
print("Maximum sentence length:", max(sentence_lengths))
print("Average sentence length:", np.mean(sentence_lengths))

Minimum sentence length: 1
Maximum sentence length: 271
Average sentence length: 25.722023505365357


In [14]:
## 4. Data Preprocessing
def preprocess_sentence(sentence):
    """
    Convert a Penn Treebank sentence into normalized tokens.
    """
    return [word.lower() for word in sentence]

sample_sentence = sentences[0]

print("Original:")
print(sample_sentence)

print("\nPreprocessed:")
print(preprocess_sentence(sample_sentence))

Original:
['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'Nov.', '29', '.']

Preprocessed:
['pierre', 'vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'nov.', '29', '.']


In [16]:
processed_sentences = [
    preprocess_sentence(sentence)
    for sentence in sentences
]

print("Number of processed sentences:", len(processed_sentences))
print(processed_sentences[0])

Number of processed sentences: 3914
['pierre', 'vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'nov.', '29', '.']


In [17]:
processed_sentences = [
    sentence for sentence in processed_sentences
    if len(sentence) > 0
]

print("Number of non-empty sentences:", len(processed_sentences))

Number of non-empty sentences: 3914


In [18]:
random.seed(SEED)

indices = list(range(len(processed_sentences)))
random.shuffle(indices)

split_idx = int(0.8 * len(indices))

train_indices = indices[:split_idx]
test_indices = indices[split_idx:]

train_sentences = [processed_sentences[i] for i in train_indices]
test_sentences = [processed_sentences[i] for i in test_indices]

In [20]:
print("Training sentences:", len(train_sentences))
print("Testing sentences :", len(test_sentences))
# percentage
print("Training percentage:", len(train_sentences) / len(processed_sentences) * 100)
print("Testing percentage :", len(test_sentences) / len(processed_sentences) * 100)

Training sentences: 3131
Testing sentences : 783
Training percentage: 79.99489013796628
Testing percentage : 20.005109862033727


In [21]:
print("Sample training sentence:")
print(train_sentences[0])

print("\nSample test sentence:")
print(test_sentences[0])

Sample training sentence:
['the', 'company', 'noted', 'that', 'it', 'has', 'reduced', 'debt', 'by', '$', '1.6', 'billion', '*u*', 'since', 'the', 'end', 'of', '1988', 'and', 'bought', 'back', 'about', '15.5', 'million', 'shares', 'of', 'common', 'stock', 'since', 'the', 'fourth', 'quarter', 'of', '1987', '.']

Sample test sentence:
['terms', 'were', "n't", 'disclosed', '*-1', '.']


In [24]:
train_tokens = [
    token
    for sentence in train_sentences
    for token in sentence
]

test_tokens = [
    token
    for sentence in test_sentences
    for token in sentence
]

print("Training tokens:", len(train_tokens))
print("Testing tokens :", len(test_tokens))
print(train_tokens[:50])

Training tokens: 81021
Testing tokens : 19655
['the', 'company', 'noted', 'that', 'it', 'has', 'reduced', 'debt', 'by', '$', '1.6', 'billion', '*u*', 'since', 'the', 'end', 'of', '1988', 'and', 'bought', 'back', 'about', '15.5', 'million', 'shares', 'of', 'common', 'stock', 'since', 'the', 'fourth', 'quarter', 'of', '1987', '.', 'boeing', 'co.', 'said', '0', 'it', 'is', 'discussing', 'plans', '*ich*-1', 'with', 'three', 'of', 'its', 'regular', 'japanese']


In [25]:
unique_train_words = set(train_tokens)

print("Total training tokens:", len(train_tokens))
print("Unique training words:", len(unique_train_words))

Total training tokens: 81021
Unique training words: 10108


In [26]:
unique_test_words = set(test_tokens)

unseen_test_words = unique_test_words - unique_train_words

print("Unique test words:", len(unique_test_words))
print("Test words unseen during training:", len(unseen_test_words))

Unique test words: 4607
Test words unseen during training: 1279


In [27]:
print("=" * 50)
print("DATA PREPROCESSING SUMMARY")
print("=" * 50)

print(f"Total sentences      : {len(processed_sentences)}")
print(f"Training sentences   : {len(train_sentences)}")
print(f"Testing sentences    : {len(test_sentences)}")
print(f"Training tokens      : {len(train_tokens)}")
print(f"Testing tokens       : {len(test_tokens)}")
print(f"Unique train words   : {len(unique_train_words)}")
print(f"Unique test words    : {len(unique_test_words)}")
print(f"Unseen test words    : {len(unseen_test_words)}")

DATA PREPROCESSING SUMMARY
Total sentences      : 3914
Training sentences   : 3131
Testing sentences    : 783
Training tokens      : 81021
Testing tokens       : 19655
Unique train words   : 10108
Unique test words    : 4607
Unseen test words    : 1279
